# Prediction-Powered Inference for AI-Driven Materials Discovery
**Companion notebook to the paper.** Runs on a laptop; no GPU or DFT required.

Two parts:
1. **Synthetic validation (runs immediately, offline):** verifies coverage, width, and bias behavior of every estimator on data where the ground truth is known by construction.
2. **Real-data pipeline (Matbench Discovery / WBM):** `>>> TO RUN:` cells that download published model predictions and DFT labels, then reproduce every table and figure of the paper. All numbers in the paper are `[RESULT PLACEHOLDER]` until you run these.

**Integrity note:** this code *computes* results; it never asserts them. If the real-data results contradict the paper's expected findings, the paper's claims must be revised, not the code.

In [1]:
# ---- Environment self-check (run me first) ----
import sys
print(f"Python {sys.version.split()[0]} | kernel: {sys.executable}")
_env_ok = True
try:
    import numpy as _np, pandas as _pd
    print(f"numpy {_np.__version__} | pandas {_pd.__version__} -- imports OK")
except Exception as _err:
    _env_ok = False
    print("ENVIRONMENT PROBLEM:", type(_err).__name__, str(_err)[:150])
try:
    import matbench_discovery as _mbd
    print(f"matbench-discovery {_mbd.__version__} -- import OK")
except Exception as _err:
    _env_ok = False
    print("matbench-discovery import FAILED:", type(_err).__name__, str(_err)[:120])
try:
    import pymatgen.core as _pmg  # the usual legacy culprit in Anaconda base
    print("pymatgen -- import OK")
except Exception as _err:
    _env_ok = False
    print("pymatgen import FAILED (this is the np.unicode_/_ARRAY_API culprit):",
          type(_err).__name__, str(_err)[:120])
_in_base = ("anaconda" in sys.executable.lower()
            and "envs" not in sys.executable.lower())
if _in_base:
    print()
    print("WARNING: this kernel is the Anaconda BASE environment"
          " (no 'envs/' in the path above).")
    print("Even if you created the certdisc env, THIS notebook is not using it.")
    print("Fix: Kernel -> Change Kernel -> certified-discovery. If that kernel")
    print("is missing or still shows base, re-register it FROM INSIDE the env:")
    print("    conda activate certdisc")
    print("    python -m ipykernel install --user --name certdisc \\")
    print("        --display-name certified-discovery")
if (not _env_ok) or _in_base:
    print()
    print("You appear to be in the Anaconda BASE environment (or imports failed).")
    print("Base mixes packages compiled against different NumPy versions, which")
    print("causes '_ARRAY_API not found' / 'np.unicode_' errors. Fix (Terminal):")
    print("    conda create -n certdisc python=3.11 -y")
    print("    conda activate certdisc")
    print("    pip install -r requirements.txt")
    print("    python -m ipykernel install --user --name certdisc \\")
    print("        --display-name certified-discovery")
    print("then in Jupyter: Kernel -> Change Kernel -> certified-discovery,")
    print("and re-run this notebook top to bottom.")

Python 3.12.3 | kernel: /usr/bin/python3


numpy 2.4.4 | pandas 3.0.2 -- imports OK


matbench-discovery 1.3.1 -- import OK
pymatgen -- import OK


In [2]:
# --- bootstrap: this notebook is self-contained -----------------------------
# The analysis uses the ppi_matdisc toolkit. If ppi_matdisc.py is not found in
# the working directory, an exact embedded copy is written to disk first (the
# sha256 is printed for provenance). An existing ppi_matdisc.py next to the
# notebook is NEVER overwritten -- the repository copy always wins.
import os, sys, base64, hashlib, pathlib
WORKDIR = "/Users/hasankurban/Downloads/files-4"   # <- all outputs land here; edit if you move the folder
try:
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
except Exception as _e:
    print(f"note: could not switch to {WORKDIR} ({type(_e).__name__}); staying in {os.getcwd()}")
print("working directory:", os.getcwd())
os.makedirs("results", exist_ok=True)
_here = pathlib.Path.cwd()
_EMBEDDED_SHA256 = "b781d635df82ef906e8965bf115e585cdd034e5997d1c13ee031b1fe79db61f9"
_EMBEDDED_PPI_MATDISC_B64 = (
    "..."
)
if (_here / "ppi_matdisc.py").is_file():
    print("using existing ppi_matdisc.py found next to the notebook")
else:
    _raw = base64.b64decode(_EMBEDDED_PPI_MATDISC_B64)
    assert hashlib.sha256(_raw).hexdigest() == _EMBEDDED_SHA256, "embedded toolkit corrupted"
    (_here / "ppi_matdisc.py").write_bytes(_raw)
    print(f"ppi_matdisc.py not found -> wrote embedded copy "
          f"({len(_raw):,} bytes, sha256 {_EMBEDDED_SHA256[:16]}...)")
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ppi_matdisc import (classical_ci, imputation_estimate, ppi_mean_ci,
                         ppi_count_ci, ppi_yield_diff_ci, neyman_allocation,
                         stratified_ppi_ci, simulate_campaign, budget_curve,
                         required_budget, wilson_ci)
np.random.seed(0)
print("imports ok")


imports ok


## How to run this notebook cell by cell
Use **Shift+Enter** to run the selected cell and move to the next. Run **top to
bottom**; every cell below states what it needs, and the experiment cells will
tell you which cell to run first if anything is missing (they never crash from
running out of order).

| # | Cell | What it does | Needs | Time (pilot / full) |
|---|------|--------------|-------|---------------------|
| 1 | imports | loads the toolkit | -- | seconds |
| 2--4 | Part 1: synthetic control | validates the estimators where truth is known; prints the numbers quoted in the paper | cell 1 | ~1 min |
| 5 | Figure 2 | regenerates the paper's four-panel figure | cells 1--4 | ~2 min |
| 6 | stratified demo | synthetic Neyman allocation | cells 1--4 | seconds |
| 7 | **CONFIG** | the only cell you edit (`QUICK_MODE`, models, budgets) | -- | instant |
| 8 | **DATA LOADING** | downloads/caches WBM + predictions, reports which models it found | 7 | ~3 min first run |
| 9 | **E1** counts | fills Table 1 + Supplementary Table 1; prints LaTeX rows | 1, 7, 8 | min / ~30 min |
| 10 | **E2** budgets | fills Fig. 3 + Corollary-1 check | 1, 7, 8, 9 (reuses `campaign_arrays`) | min / ~15 min |
| 11 | **E3** rankings | fills Table 2; prints LaTeX rows | 1, 7, 8 | min / ~15 min |
| 12 | **E4/E5** | fills Supplementary Table 2 + robustness | 1, 7, 8, 9 | min / ~10 min |

Workflow: run 1--12 once with `QUICK_MODE=True` (sanity pilot; its LaTeX rows
are stamped *PILOT -- not for the paper*), then set `QUICK_MODE=False` in the
CONFIG cell and re-run cells 7--12 for the paper numbers.

## Part 1 — Synthetic validation (ground truth known)
We simulate a screening campaign: a pool of $N$ candidates flagged stable by a model, a true stability label per candidate, and an imperfect, optimistically biased model. We then hide all but $n$ labels and check that (a) classical and PPI++ intervals both achieve nominal coverage, (b) PPI++ is never materially wider, and (c) the naive "trust the model" count is biased.

In [3]:
rng = np.random.default_rng(7)
N = 20000
latent = rng.normal(0, 1, N)
p_true = 1 / (1 + np.exp(-(latent - 0.4) * 2.2))
y = rng.binomial(1, p_true).astype(float)          # "DFT" ground truth
yhat = ((latent + rng.normal(0, 0.9, N)) > -0.25).astype(float)  # model call

truth = y.mean()
print(f"true precision {truth:.4f} | naive estimate {imputation_estimate(yhat):.4f} "
      f"| naive bias {imputation_estimate(yhat)-truth:+.4f}")
res = simulate_campaign(y, yhat, n_label=300, n_rep=400)
for k, v in res.items():
    print(f"{k:>24}: {v}")
assert res["coverage_classical"] > 0.9 and res["coverage_ppi"] > 0.9

true precision 0.3754 | naive estimate 0.5729 | naive bias +0.1975
                   truth: 0.37535
     imputation_estimate: 0.5729
         imputation_bias: 0.19754999999999995
      coverage_classical: 0.955
            coverage_ppi: 0.9525
    mean_width_classical: 0.10883715081449101
          mean_width_ppi: 0.10115986071053026
         width_reduction: 0.07053924185360572
                 n_label: 300
                   n_rep: 400
                   alpha: 0.05


### Paper Figure 2: four-panel synthetic validation
Panel a contrasts naive/classical/certified at one budget; panels b-c give coverage and width against budget; panel d tests the sqrt(1-rho^2) efficiency law of Theorem 1. The real-data analogues replace `y, yhat` with WBM labels and uMLIP predictions.

In [4]:
# ---- Figure 2 (paper): four-panel synthetic validation, Nature styling ----
OKB, OKO, OKG, OKV, OKGR = "#0072B2", "#E69F00", "#009E73", "#D55E00", "#8C8C8C"
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "Arial", "DejaVu Sans"], "font.size": 7.0,
    "axes.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "legend.frameon": False, "axes.titlesize": 7.5,
    "axes.labelsize": 7.0, "xtick.labelsize": 7.0, "ytick.labelsize": 7.0,
})
fig, axes = plt.subplots(2, 2, figsize=(7.09, 4.9))
(a, b), (c, d) = axes

# -- panel a: naive vs classical vs certified at n = 300 --
rng_a = np.random.default_rng(11)
idx = rng_a.choice(N, 300, replace=False)
cl = classical_ci(y[idx], pool_size=N)
pp = ppi_mean_ci(y[idx], yhat[idx], yhat)
naive = imputation_estimate(yhat)
a.axhline(truth, color="k", lw=0.8, ls=(0, (4, 3)))
a.text(2.62, truth - 0.012, "true precision", va="top", fontsize=6.5)
a.errorbar([1], [cl.estimate], yerr=[[cl.estimate - cl.lower], [cl.upper - cl.estimate]],
           fmt="o", color=OKGR, ms=4, capsize=3, lw=1.1)
a.errorbar([2], [pp.estimate], yerr=[[pp.estimate - pp.lower], [pp.upper - pp.estimate]],
           fmt="o", color=OKB, ms=4, capsize=3, lw=1.1)
a.plot([3], [naive], "s", color=OKV, ms=5)
a.annotate("", xy=(3, naive), xytext=(3, truth),
           arrowprops=dict(arrowstyle="-|>", color=OKV, lw=0.9))
a.text(2.86, (naive + truth) / 2, f"+{naive-truth:.2f}\nbias", color=OKV, fontsize=6.5, ha="right")
a.text(3.1, naive + 0.002, "no\ninterval", ha="left", va="center", color=OKV, fontsize=6.5)
a.set_ylim(min(cl.lower, pp.lower) - 0.03, naive + 0.045)
a.set_xticks([1, 2, 3]); a.set_xticklabels(["classical\n(labels only)", "certified\n(this work)", "naive\n(trust model)"])
a.set_xlim(0.5, 3.7); a.set_ylabel("estimated precision"); a.set_title("Point estimates and 95% intervals, n = 300", loc="left", pad=8)

# -- panels b, c: coverage and width vs budget --
budgets = [50, 100, 200, 400, 800, 1600]
curve = budget_curve(y, yhat, budgets, n_rep=500)
mc = 1.96 * np.sqrt(0.95 * 0.05 / 500)
b.axhspan(0.95 - mc, 0.95 + mc, color="k", alpha=0.08, lw=0)
b.axhline(0.95, color="k", lw=0.8, ls=(0, (4, 3)))
b.plot(budgets, [q["coverage_classical"] for q in curve], "o-", color=OKGR, ms=3.5, lw=1.1)
b.plot(budgets, [q["coverage_ppi"] for q in curve], "o-", color=OKB, ms=3.5, lw=1.1)
b.text(budgets[-1], 0.9735, "nominal 0.95 (band: Monte-Carlo error)", ha="right", va="bottom", fontsize=6.2)
b.text(55, 0.912, "classical", color=OKGR, fontsize=6.8)
b.text(55, 0.897, "certified", color=OKB, fontsize=6.8)
b.set_xscale("log"); b.set_ylim(0.88, 1.005)
b.set_xlabel("DFT verification budget n"); b.set_ylabel("empirical coverage")
b.set_title("Coverage holds at every budget", loc="left")

wc = np.array([q["mean_width_classical"] for q in curve])
wp = np.array([q["mean_width_ppi"] for q in curve])
c.fill_between(budgets, wp, wc, color=OKB, alpha=0.12, lw=0)
c.plot(budgets, wc, "o-", color=OKGR, ms=3.5, lw=1.1)
c.plot(budgets, wp, "o-", color=OKB, ms=3.5, lw=1.1)
c.text(600, 0.212, "classical", color=OKGR, fontsize=6.8)
c.text(600, 0.186, "certified", color=OKB, fontsize=6.8)
_mid65 = (np.interp(np.log(65), np.log(budgets), wc)
          + np.interp(np.log(65), np.log(budgets), wp)) / 2
c.annotate("prediction gain", xy=(65, _mid65), xytext=(135, 0.252),
           color=OKB, fontsize=6.4, va="center",
           arrowprops=dict(arrowstyle="->", color=OKB, lw=0.7,
                           connectionstyle="arc3,rad=0.18", shrinkA=2, shrinkB=1))
c.set_xscale("log"); c.set_xlabel("DFT verification budget n"); c.set_ylabel("mean 95% CI width")
c.set_title("Predictions stretch the same budget", loc="left")

# -- panel d: width ratio versus rho, against the sqrt(1 - rho^2) law --
def _sweep(make_f, sigmas):
    rr, qq = [], []
    for sigma in sigmas:
        f_s = make_f(sigma)
        rr.append(float(np.corrcoef(y, f_s)[0, 1]))
        res_s = simulate_campaign(y, f_s, n_label=300, n_rep=150, seed=3)
        qq.append(res_s["mean_width_ppi"] / res_s["mean_width_classical"])
    return rr, qq
sig = [0.02, 0.25, 0.6, 1.0, 1.8, 3.5]
rhos, ratios = _sweep(lambda s_: ((latent + rng_a.normal(0, s_, N)) > -0.25).astype(float), sig)
rhos_m, ratios_m = _sweep(lambda s_: 1.0 / (1.0 + np.exp(-1.8 * (latent + rng_a.normal(0, s_, N) - 0.4))), sig)
grid = np.linspace(0, 1, 200)
d.plot(grid, np.sqrt(1 - grid ** 2), "k-", lw=1.0)
d.plot(rhos, ratios, "o", color=OKB, ms=4)
d.plot(rhos_m, ratios_m, "s", mfc="none", mec=OKG, mew=1.1, ms=4.5)
# overlay the ten real WBM campaigns if a completed run's summary is present
try:
    import pandas as _pd
    _bs = _pd.read_csv("results/budgets_summary.csv")
    if "rho_pool" in _bs.columns:
        d.plot(_bs.rho_pool, _bs.ratio_n400, "D", color=OKV, ms=4.2,
               mec="white", mew=0.5, zorder=4)
        d.text(0.055, 0.845, "real campaigns\n(ten models, WBM)", color=OKV,
               fontsize=6.5, ha="left", va="top")
except Exception:
    pass  # first pass before E2: synthetic-only panel
d.text(0.26, 0.985, "binary calls", color=OKB, fontsize=6.5)
d.text(0.68, 0.775, "continuous margins", color=OKG, fontsize=6.5, ha="left", va="bottom")
d.text(0.04, 0.545, r"$\sqrt{1-\rho^2}$" + "\n(Theorem 1, known variances)", fontsize=6.5, va="bottom")
d.set_xlim(0, 1); d.set_ylim(0.5, 1.05)
d.set_xlabel(r"prediction–label correlation $\rho$ on the pool")
d.set_ylabel("width ratio, certified / classical")
d.set_title("Efficiency follows the predicted law", loc="left")

for ax, letter in zip([a, b, c, d], "abcd"):
    ax.text(-0.16, 1.08, letter, transform=ax.transAxes, fontsize=10, fontweight="bold")
fig.tight_layout(w_pad=2.2, h_pad=2.4)
os.makedirs("results", exist_ok=True)
fig.savefig("results/fig2_synthetic_validation.png", dpi=400, bbox_inches="tight")

# persist every caption-relevant quantity so the results zip is self-sufficient
_rows = [("panel_a", "true_precision", truth),
         ("panel_a", "naive_estimate", naive),
         ("panel_a", "naive_bias", naive - truth),
         ("panel_a", "classical_lo", cl.lower), ("panel_a", "classical_est", cl.estimate),
         ("panel_a", "classical_hi", cl.upper),
         ("panel_a", "certified_lo", pp.lower), ("panel_a", "certified_est", pp.estimate),
         ("panel_a", "certified_hi", pp.upper)]
_rows += [("panel_a_mc400", k, float(v)) for k, v in res.items() if np.isscalar(v)]
for _s, _r, _q in zip(sig, rhos, ratios):
    _rows += [("panel_d_binary", f"rho@sigma={_s}", _r), ("panel_d_binary", f"ratio@sigma={_s}", _q)]
for _s, _r, _q in zip(sig, rhos_m, ratios_m):
    _rows += [("panel_d_margins", f"rho@sigma={_s}", _r), ("panel_d_margins", f"ratio@sigma={_s}", _q)]
pd.DataFrame(_rows, columns=["section", "key", "value"]).to_csv(
    "results/fig2_synthetic_summary.csv", index=False)
pd.DataFrame(curve).to_csv("results/fig2_budget_curve_synthetic.csv", index=False)
print("fig2 saved -> results/ (png + summary csv + budget-curve csv)")
print("saved fig2_synthetic_validation.png; rho sweep:",
      [f"{r:.2f}/{q:.2f}" for r, q in zip(rhos, ratios)])

/tmp/ipykernel_935/223486166.py:104: UserWarning: The figure layout has changed to tight
  fig.tight_layout(w_pad=2.2, h_pad=2.4)


saved fig2_synthetic_validation.png; rho sweep: ['0.51/0.86', '0.49/0.87', '0.43/0.90', '0.34/0.94', '0.24/0.97', '0.14/0.99']


### Stratified PPI with Neyman DFT-budget allocation (paper Sec. 2.4)

In [5]:
strata_id = (latent > 0).astype(int)
sizes = [int((strata_id == k).sum()) for k in (0, 1)]
pilot = rng.choice(N, 100, replace=False)
sds = []
for k in (0, 1):
    m = strata_id[pilot] == k
    r = y[pilot][m] - yhat[pilot][m]
    sds.append(r.std(ddof=1) if m.sum() > 1 else 1.0)
alloc = neyman_allocation(sizes, sds, budget=300)
print("allocation:", alloc)
strata = []
for k in (0, 1):
    ids = rng.choice(np.where(strata_id == k)[0], alloc[k], replace=False)
    strata.append({"y": y[ids], "f_l": yhat[ids], "f_pool": yhat[strata_id == k]})
print(stratified_ppi_ci(strata), "| truth", round(truth, 4))

allocation: [143 157]
stratified PPI++: 0.3640 [0.3217, 0.4062] (1-alpha=0.95, n=300) | truth 0.3754


## Part 2 — Real data: turnkey pipeline for Matbench Discovery / WBM

**Run this on your machine** (internet required on first run; ~200 MB cached download):

```bash
pip install -U matbench-discovery pandas matplotlib jupyter
jupyter lab ppi_matdisc_analysis.ipynb   # then: Run All
```

How it works:
1. The **CONFIG** cell is the only thing you edit. `QUICK_MODE=True` (default) runs a fast pilot (minutes) so you can verify the pipeline end to end; set it to `False` for the paper protocol (1,000 replicates; expect on the order of an hour on a laptop).
2. The **loader** uses the official `matbench_discovery` API, resolves the requested model list against whatever your installed version provides, and reports exactly which models it found. If the current-leaderboard models are missing, upgrade the package (`pip install -U matbench-discovery`, or install from the GitHub main branch).
3. Each experiment cell (E1–E5) writes a CSV under `results/`, prints the table in the paper's exact column order, and E1/E3 also print ready-to-paste LaTeX rows for Tables 1–2.
4. If data is unavailable (e.g., offline), every cell degrades to a printed instruction instead of an error.

In [6]:
# ============================ CONFIG (edit me) ============================
QUICK_MODE   = False         # False: paper protocol (default). True: fast pilot.
# OPTION B (2026 leaderboard). When the matbench-discovery PACKAGE is
# installed, the loader AUTO-DISCOVERS every model your installed version
# ships and runs all of them -- no need to know the exact leaderboard
# labels in advance. Set MODELS = [...] to a specific list only if you want
# to restrict to particular models; leave it None to run the full current
# leaderboard. Without the package, the loader falls back to the pinned
# v1.3.1 ten-model generation (the paper's run of record).
MODELS       = [
    'EquiformerV3+DeNS-OAM',
    'eSEN-30M-OAM',
    'PET-OAM-XL',
    'GRACE-2L-OAM',
    'ORB v3',
    'SevenNet-MF-ompa',
    'MACE-MPA-0',
    'MatRIS-10M-OAM',
    'DPA3-v1-OpenLAM',
    'MatterSim v1 5M',
    'CHGNet',
    'M3GNet',
    'ALIGNN',
    'MEGNet',
    'CGCNN',
]  # curated Option B set: 2026 frontier + paper continuity anchors.
                             # Set MODELS=None to run ALL models the package exposes.
MAX_MODELS   = None          # e.g. 20 to cap runtime; None => no cap
PREFER_PACKAGE = True        # False forces the pinned v1.3.1 fallback even if the package is present
TAUS         = [0.0, 0.05, 0.10]  # stability thresholds, eV/atom
POOLS        = ["full", "uniq_protos"]
N_LABEL      = 500                # verification budget per campaign (Table 1)
N_REP        = 50 if QUICK_MODE else 1000
BUDGETS      = [50, 100, 200, 400, 800] if QUICK_MODE else [50, 100, 200, 400, 800, 1600, 3200]
ALPHA        = 0.05
BASE_SEED    = 0
SEED_ROBUST  = [1, 2, 3, 4]   # E6: extra base seeds, headline slice robustness
RANK_BUDGET2 = 2000           # E3: second, larger budget for ranking sensitivity
MARGIN_SCALE = 0.05  # eV/atom; monotone margin transform f = sigmoid(-(E_hull_pred - tau)/scale).
                     # Inside a single model's flagged pool the binary call is
                     # identically 1 and carries no information, so margins are the
                     # default; validity never depends on this choice (Theorem 1),
                     # only interval width does. E5 checks binary calls explicitly.
import os
os.makedirs("results", exist_ok=True)
NOTEBOOK_BUILD = "2026-07-03-optionB-colfix-r6"
print(f"config ok | build={NOTEBOOK_BUILD} | QUICK_MODE={QUICK_MODE} | N_REP={N_REP}")

config ok | QUICK_MODE=False | N_REP=1000


In [7]:
# ============== DATA LOADING: package first, direct-download fallback ==============
# Path 1 uses the matbench-discovery package (newest models; needs a clean env).
# Path 2 needs ONLY numpy+pandas: it downloads the published WBM summary and
# model prediction files straight from the matbench-discovery GitHub repository
# (tag v1.3.1) -- every URL and column name below was verified against the real
# files. Path 2 works even in environments where the package cannot import.
import os, urllib.request
import numpy as np, pandas as pd

DATA_OK, DF, EACH_PRED, FOUND, LOAD_PATH = False, {}, {}, [], None
truth_col = "e_above_hull_mp2020_corrected_ppd_mp"
_EFORM_DFT = "e_form_per_atom_mp2020_corrected"

def _try_package():
    global DF, EACH_PRED, FOUND
    if not PREFER_PACKAGE:
        raise RuntimeError("PREFER_PACKAGE is False; using pinned fallback")
    from matbench_discovery.preds import load_df_wbm_with_preds
    from matbench_discovery.enums import Model, MbdKey
    # Enumerate models robustly: iterate the Model enum rather than reading
    # Model.label_map, because a single malformed metadata YAML (e.g. a newly
    # added model) makes label_map raise while iteration still succeeds. Each
    # member's string label is what load_df_wbm_with_preds expects.
    installed, _menum_of = [], {}
    for _m in Model:
        try:
            _lab = getattr(_m, "label", None) or str(_m.value)
            installed.append(_lab); _menum_of[_lab] = _m
        except Exception:
            continue
    installed = list(dict.fromkeys([x for x in installed if x]))  # dedupe, drop blanks
    # choose the model set: explicit MODELS if given, else the full leaderboard
    if MODELS:
        wanted = [m for m in MODELS if m in installed]
        missing = [m for m in MODELS if m not in installed]
        if missing:
            print(f"note: requested but not in your installed package: {missing}")
        if not wanted:
            raise RuntimeError("none of the requested MODELS are in the installed package")
    else:
        wanted = installed
        print(f"auto-discovered {len(wanted)} models in installed package (Option B, full leaderboard)")
    if MAX_MODELS:
        wanted = wanted[:MAX_MODELS]
        print(f"capped to first {len(wanted)} models via MAX_MODELS")
    # Load per model. The diagnostic confirmed the prediction column is named
    # EXACTLY by the display label (e.g. "EquiformerV3+DeNS-OAM", float64), so
    # dfm[m] is the prediction. We still coerce and guard, and we PRINT any real
    # exception per model so failures are never silent.
    _META = {MbdKey.each_true, MbdKey.e_form_dft, "formula",
             "unique_prototype", "material_id", "n_sites"}
    import traceback as _tb
    for pool in POOLS:
        # NB: the installed package raises KeyError on subset="uniq_protos"
        # (it wants an enum/Index, not that string). The frame already carries
        # a boolean "unique_prototype" column (confirmed by diagnostic), so we
        # load the FULL frame and filter by that column for the uniq_protos pool.
        want_uniq = (pool == "uniq_protos")
        loaded, cols_ok = None, []
        for m in wanted:
            try:
                dfm = load_df_wbm_with_preds(models=[m], subset=None, pbar=False)
                if want_uniq:
                    if "unique_prototype" not in dfm.columns:
                        if m == wanted[0]:
                            print("  uniq_protos pool: no unique_prototype column; skipping this pool")
                        break
                    dfm = dfm[dfm["unique_prototype"].astype(bool)]
                # locate the prediction column: exact label match first
                if m in dfm.columns:
                    pred = pd.to_numeric(dfm[m], errors="coerce")
                else:
                    # fallback: the one non-metadata numeric column that appeared
                    cand = [col for col in dfm.columns if col not in _META]
                    pred = None
                    for col in cand:
                        v = pd.to_numeric(dfm[col], errors="coerce")
                        if v.notna().sum() > 0:
                            pred = v; break
                if pred is None or pred.notna().sum() == 0:
                    print(f"  skip {m}: no numeric prediction column in {list(dfm.columns)}")
                    continue
                if loaded is None:
                    keep = [c for c in (MbdKey.each_true, MbdKey.e_form_dft,
                                        "formula", "unique_prototype") if c in dfm.columns]
                    loaded = dfm[keep].copy()
                loaded[m] = pred.to_numpy()
                cols_ok.append(m)
            except Exception as e:
                print(f"  skip {m}: {type(e).__name__}: {str(e)[:120]}")
                if pool == POOLS[0] and m == wanted[0]:
                    _tb.print_exc()   # full trace for the first failure only
        if loaded is None:
            raise RuntimeError("package present but no model predictions loaded")
        loaded = loaded.rename(columns={MbdKey.each_true: truth_col,
                                        MbdKey.e_form_dft: _EFORM_DFT})
        DF[pool] = loaded
        EACH_PRED[pool] = {m: loaded[truth_col] + loaded[m] - loaded[_EFORM_DFT]
                           for m in cols_ok}
    FOUND[:] = [m for m in wanted if m in EACH_PRED[POOLS[0]]]
    print(f"  package models loaded ({len(FOUND)}): {FOUND}")
    return "matbench-discovery package"

_FB_BASE = "https://github.com/janosh/matbench-discovery/raw/v1.3.1"
_FB_WBM = "data/wbm/2023-12-13-wbm-summary.csv.gz"
_FB_MODELS = {  # label -> repo-relative prediction file (v1.3.1 generation)
    "CHGNet": "models/chgnet/2023-12-21-chgnet-0.3.0-wbm-IS2RE.csv.gz",
    "MACE": "models/mace/2023-12-11-mace-wbm-IS2RE-FIRE.csv.gz",
    "SevenNet": "models/sevennet/2024-07-11-sevennet-preds.csv.gz",
    "ORB": "models/orb/orbff-v1-20240827.csv.gz",
    "ORB MPtrj": "models/orb/orbff-mptrj-only-v1-20240827.csv.gz",
    "MatterSim": "models/mattersim/mattersim-wbm-IS2RE.csv.gz",
    "M3GNet": "models/m3gnet/2023-12-28-m3gnet-wbm-IS2RE.csv.gz",
    "GNoME": "models/gnome/2023-11-01-gnome-preds-50076332.csv.gz",
    "MEGNet": "models/megnet/2022-11-18-megnet-wbm-IS2RE.csv.gz",
    "ALIGNN": "models/alignn/2023-06-02-alignn-wbm-IS2RE.csv.gz",
    "Wrenformer": "models/wrenformer/2022-11-15-wrenformer-ens=10-IS2RE-preds.csv.gz",
    "CGCNN": "models/cgcnn/2023-01-26-cgcnn-ens=10-wbm-IS2RE.csv.gz",
}

def _fetch(rel):
    os.makedirs("mbd_cache", exist_ok=True)
    dest = os.path.join("mbd_cache", rel.replace("/", "__"))
    if not os.path.isfile(dest):
        print(f"  downloading {rel} ...")
        urllib.request.urlretrieve(f"{_FB_BASE}/{rel}", dest)
    return dest

def _try_fallback():
    global DF, EACH_PRED, FOUND
    wbm = pd.read_csv(_fetch(_FB_WBM)).set_index("material_id")
    frames = {"full": wbm}
    if "unique_prototype" in wbm.columns:
        frames["uniq_protos"] = wbm[wbm["unique_prototype"].astype(bool)]
    else:
        print("  note: no unique_prototype column; uniq_protos pool unavailable")
    found = []
    preds = {}
    for label, rel in _FB_MODELS.items():
        try:
            m = pd.read_csv(_fetch(rel)).set_index("material_id")
        except Exception as e:
            print(f"  skip {label}: {type(e).__name__}"); continue
        cand = [c for c in m.columns if c.startswith("e_form_per_atom")]
        if len(cand) != 1:  # ensemble files publish one aggregated column
            cand = [c for c in m.columns if c.endswith("_pred_ens")]
        if len(cand) != 1:
            print(f"  skip {label}: ambiguous prediction columns {list(m.columns)}"); continue
        preds[label] = m[cand[0]]; found.append(label)
    if not found:
        raise RuntimeError("fallback: no model files loadable")
    for pool, base in frames.items():
        DF[pool] = base
        EACH_PRED[pool] = {}
        for label in found:
            p = preds[label].reindex(base.index)
            # replicate the package's max_error_threshold=5 eV/atom filter:
            # a handful of corrupt predictions otherwise poison MAEs and pools
            err = (p - base[_EFORM_DFT]).abs()
            p = p.where(err <= 5.0)
            EACH_PRED[pool][label] = base[truth_col] + p - base[_EFORM_DFT]
    if pool_diff := [p for p in POOLS if p not in frames]:
        print(f"  pools unavailable in fallback: {pool_diff}")
    FOUND[:] = found
    if MODELS:
        req = set(MODELS) & set(found)
        print(f"  requested-and-available: {sorted(req) if req else 'none of the CONFIG list'}")
    print(f"  fallback model generation (v1.3.1): {found}")
    print("  -> Table 1's row set follows the models actually run (the caption")
    print("     already states this); a clean env + upgraded package unlocks the")
    print("     newest leaderboard models.")
    return "direct GitHub download (package-free)"

for _name, _fn in [("package", _try_package), ("direct-download fallback", _try_fallback)]:
    try:
        LOAD_PATH = _fn(); DATA_OK = True
        print(f"loaded via {LOAD_PATH}: " +
              ", ".join(f"{p}={len(DF[p]):,} rows" for p in DF) +
              f" | models: {FOUND}")
        break
    except Exception as err:
        print(f"{_name} path failed: {type(err).__name__}: {str(err)[:160]}")

if not DATA_OK:
    print()
    print("Both loading paths failed. If errors mention '_ARRAY_API' or")
    print("'np.unicode_', your kernel mixes NumPy generations (Anaconda base):")
    print("create the certdisc env per the first cell's instructions. If errors")
    print("are network-related, connect to the internet and re-run this cell.")
MBD_VERSION = None
try:
    import matbench_discovery as _mbd
    MBD_VERSION = getattr(_mbd, "__version__", "installed (version attr missing)")
except Exception:
    pass
if DATA_OK and LOAD_PATH and LOAD_PATH.startswith("direct"):
    # pinned-fallback path: pool census must match the paper's run of record
    _expected = {"full": 256963, "uniq_protos": 215488}
    for _p, _n in _expected.items():
        if _p in DF and len(DF[_p]) != _n:
            print(f"WARNING: {_p} pool has {len(DF[_p]):,} rows; the paper's pinned "
                  f"v1.3.1 run had {_n:,}. Do not paste numbers before resolving.")
elif DATA_OK and LOAD_PATH and LOAD_PATH.startswith("matbench"):
    # Option B: newer package generation. Pool size may differ from v1.3.1
    # (the WBM test corpus is periodically revised); this is expected, not an
    # error. The census the paper reports is whatever THIS run measured; the
    # metadata records the version and every model actually loaded.
    print(f"Option B run: {len(FOUND)} models on pools " +
          ", ".join(f"{p}={len(DF[p]):,}" for p in DF) +
          f" | matbench-discovery {MBD_VERSION}. All downstream tables and "
          "figures will reflect THIS model set; reconcile the paper to the "
          "run_metadata.json + CSVs, not to the v1.3.1 numbers.")


Error downloading url='https://figshare.com/ndownloader/files/44225498'
to file_path='/root/.cache/matbench-discovery/wbm/2023-12-13-wbm-summary.csv.gz'.
403 Client Error: Forbidden for url: https://figshare.com/ndownloader/files/44225498
package path failed: FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/matbench-discovery/wbm/2023-12-13-wbm-summary.csv.gz'


  skip GNoME: ambiguous prediction columns ['e_gnome_after_relax']


  skip Wrenformer: ambiguous prediction columns ['aleatoric_std_1', 'e_form_per_atom_mp2020_corrected_pred_1', 'aleatoric_std_2', 'e_form_per_atom_mp2020_corrected_pred_2', 'aleatoric_std_3', 'e_form_per_atom_mp2020_corrected_pred_3', 'aleatoric_std_4', 'e_form_per_atom_mp2020_corrected_pred_4', 'aleatoric_std_5', 'e_form_per_atom_mp2020_corrected_pred_5', 'aleatoric_std_6', 'e_form_per_atom_mp2020_corrected_pred_6', 'aleatoric_std_7', 'e_form_per_atom_mp2020_corrected_pred_7', 'aleatoric_std_8', 'e_form_per_atom_mp2020_corrected_pred_8', 'aleatoric_std_9', 'e_form_per_atom_mp2020_corrected_pred_9', 'aleatoric_std_10', 'e_form_per_atom_mp2020_corrected_pred_10', 'e_form_per_atom_mp2020_corrected_pred_ens', 'e_form_per_atom_mp2020_corrected_epistemic_std_ens', 'e_form_per_atom_mp2020_corrected_aleatoric_std_ens', 'e_form_per_atom_mp2020_corrected_total_std_ens', 'e_form_per_atom_wrenformer_pred_ens']


  requested-and-available: ['CHGNet']
  fallback model generation (v1.3.1): ['CHGNet', 'MACE', 'SevenNet', 'ORB', 'ORB MPtrj', 'MatterSim', 'M3GNet', 'MEGNet', 'ALIGNN', 'CGCNN']
  -> Table 1's row set follows the models actually run (the caption
     already states this); a clean env + upgraded package unlocks the
     newest leaderboard models.
loaded via direct GitHub download (package-free): full=256,963 rows, uniq_protos=215,488 rows | models: ['CHGNet', 'MACE', 'SevenNet', 'ORB', 'ORB MPtrj', 'MatterSim', 'M3GNet', 'MEGNet', 'ALIGNN', 'CGCNN']


### Experiment E1 — certified vs naive counts (fills Table 1 and Supplementary Table 1)
One campaign per (model, tau, pool): the flagged set is the pool, `N_LABEL` labels are revealed uniformly at random per replicate, and the certified interval is compared with the naive count and the census truth. Coverage below 0.93 triggers the runbook's flag-back.

In [8]:
def campaign_arrays(model, tau, pool):
    ep = EACH_PRED[pool][model]
    ok = ep.notna() & DF[pool][truth_col].notna()
    ep, truth = ep[ok].to_numpy(float), DF[pool].loc[ok, truth_col].to_numpy(float)
    flag = ep <= tau                             # model-flagged stable set S
    y = (truth[flag] <= tau).astype(float)       # census labels on S (hidden)
    f = 1.0 / (1.0 + np.exp((ep[flag] - tau) / MARGIN_SCALE))  # margin proxy
    return y, f, int(flag.sum())

def run_e1():
    rows = []
    for pool in POOLS:
        for tau in TAUS:
            for m in FOUND:
                y, f, N = campaign_arrays(m, tau, pool)
                if N < 2 * N_LABEL:
                    print(f"skip {m} tau={tau} {pool}: pool too small (N={N})"); continue
                true_ct, rng = int(y.sum()), np.random.default_rng(BASE_SEED)
                covered, cis = 0, []
                for _ in range(N_REP):
                    idx = rng.choice(N, N_LABEL, replace=False)
                    ci = ppi_mean_ci(y[idx], f[idx], f, alpha=ALPHA)
                    covered += (ci.lower <= true_ct / N <= ci.upper)
                    cis.append((ci.lower * N, ci.upper * N))
                lo, hi = np.median([c[0] for c in cis]), np.median([c[1] for c in cis])
                rows.append(dict(pool=pool, tau=tau, model=m, flagged=N,
                                 true=true_ct, ci_lo=round(lo), ci_hi=round(hi),
                                 overstate_pct=100 * (N - true_ct) / max(true_ct, 1),
                                 precision_gap=1.0 - true_ct / N,
                                 coverage=covered / N_REP))
    return pd.DataFrame(rows)

_need = [n for n in ("QUICK_MODE", "ppi_mean_ci", "DATA_OK") if n not in globals()]
if _need:
    print("Run earlier cells first (top to bottom):")
    print("  missing:", ", ".join(_need),
          "-> run the imports cell, the CONFIG cell, then the DATA LOADING cell.")
elif not DATA_OK:
    print("E1 skipped: complete the data-loading cell first.")
else:
    t1 = run_e1()
    t1.to_csv("results/table1_raw.csv", index=False)
    # Flag-back threshold scales with replicate count: 3 Monte-Carlo standard
    # errors below nominal (at N_REP=1000 this is ~0.93, matching the runbook).
    flag_thresh = 0.95 - 3 * np.sqrt(0.95 * 0.05 / N_REP)
    bad = t1[t1.coverage < flag_thresh]
    if len(bad):
        print(f"FLAG-BACK: coverage < {flag_thresh:.3f} (3 MC sigma at N_REP={N_REP})")
        print("in these configs -- STOP and investigate before proceeding:")
        print(bad[["pool", "tau", "model", "coverage"]].to_string(index=False))
    else:
        print(f"coverage check passed: all {len(t1)} configs above the "
              f"{flag_thresh:.3f} threshold (3 MC sigma at N_REP={N_REP}).")
    main_slice = t1[(t1.pool == "full") & (t1.tau == 0.0)]
    print(main_slice.to_string(index=False))
    if QUICK_MODE:
        print("\n=== PILOT MODE (QUICK_MODE=True): low-replicate sanity numbers.")
        print("=== DO NOT paste into the paper. Set QUICK_MODE=False and re-run")
        print("=== for the paper protocol before filling any red placeholder. ===")
    print("\n--- LaTeX rows for Table 1"
          + (" (PILOT -- not for the paper)" if QUICK_MODE else " (paste into main.tex)") + " ---")
    for _, r in main_slice.iterrows():
        print(f"{r.model} & {r.flagged:,} & {r.true:,} & "
              f"[{r.ci_lo:,.0f}, {r.ci_hi:,.0f}] & {r.overstate_pct:.1f} & "
              f"{r.precision_gap:.3f} & {r.coverage:.3f} \\\\")


coverage check passed: all 60 configs above the 0.929 threshold (3 MC sigma at N_REP=1000).
pool  tau     model  flagged  true  ci_lo  ci_hi  overstate_pct  precision_gap  coverage
full  0.0    CHGNet    62153 32012  29530  34421      94.155317       0.484948     0.953
full  0.0      MACE    58482 33692  31271  36013      73.578298       0.423891     0.947
full  0.0  SevenNet    53459 34581  32529  36617      54.590671       0.353130     0.965
full  0.0       ORB    38107 34258  33287  35228      11.235332       0.101005     0.954
full  0.0 ORB MPtrj    48053 34177  32443  35943      40.600404       0.288764     0.957
full  0.0 MatterSim    43740 35971  34646  37349      21.597954       0.177618     0.947
full  0.0    M3GNet    75829 33967  30820  36925     123.243148       0.552058     0.952
full  0.0    MEGNet    54380 24811  22427  27144     119.176978       0.543748     0.963
full  0.0    ALIGNN    56632 28060  25649  30494     101.824661       0.504520     0.946
full  0.0     CGCN

### Experiment E6 (strengthening) — base-seed robustness of the headline slice
Repeats the $\tau=0$ full-pool campaigns at five base seeds; coverage and width must be stable across seeds or the runbook flag-back applies.

In [ ]:
_need = [n for n in ("QUICK_MODE", "ppi_mean_ci", "DATA_OK", "campaign_arrays") if n not in globals()]
if _need or not DATA_OK:
    print("E6 skipped: run imports, CONFIG, DATA LOADING and E1 first.")
else:
    rows6 = []
    for bs in [BASE_SEED] + SEED_ROBUST:
        for m in FOUND:
            y, f, N = campaign_arrays(m, 0.0, "full")
            true_ct = int(y.sum()); rng = np.random.default_rng(bs)
            reps = max(N_REP // 2, 25); covered, widths = 0, []
            for _ in range(reps):
                idx = rng.choice(N, N_LABEL, replace=False)
                ci = ppi_mean_ci(y[idx], f[idx], f, alpha=ALPHA)
                covered += (ci.lower <= true_ct / N <= ci.upper)
                widths.append(ci.upper - ci.lower)
            rows6.append(dict(base_seed=bs, model=m, n_rep=reps,
                              coverage=covered / reps, mean_width=float(np.mean(widths))))
    d6 = pd.DataFrame(rows6); d6.to_csv("results/e6_seed_robustness.csv", index=False)
    sp = d6.groupby("model").mean_width.agg(lambda w: (w.max() - w.min()) / w.mean())
    reps = int(d6.n_rep.iloc[0])
    band = 3 * (0.95 * 0.05 / reps) ** 0.5
    n_low = int((d6.coverage < 0.95 - band).sum())
    print(f"E6: coverage across 5 seeds x {d6.model.nunique()} models in "
          f"[{d6.coverage.min():.3f}, {d6.coverage.max():.3f}]; 3-MC-sigma band at "
          f"{reps} reps is [{0.95-band:.3f}, {0.95+band:.3f}]; {n_low}/{len(d6)} below "
          f"(chance expectation at nominal coverage: ~{len(d6)*0.0013:.2f}); "
          f"max per-model width spread (max-min)/mean = {sp.max():.4f}")


### Experiment E2 — budget curves and Corollary 1 accuracy (fills Fig. 3 and the budget claims)

In [9]:
_need = [n for n in ("QUICK_MODE", "ppi_mean_ci", "DATA_OK", "campaign_arrays")
         if n not in globals()]
if _need:
    print("Run earlier cells first (top to bottom). Missing:", ", ".join(_need))
    print("  -> run imports, CONFIG, DATA LOADING, then the E1 cell (it defines")
    print("     campaign_arrays, which this cell reuses).")
elif not DATA_OK:
    print("E2 skipped: complete the data-loading cell first.")
else:
    fig_rows = []
    for m in FOUND:
        y, f, N = campaign_arrays(m, 0.0, "full")
        curve = budget_curve(y, f, [b for b in BUDGETS if 2 * b <= N], n_rep=max(N_REP // 2, 25))
        rng = np.random.default_rng(BASE_SEED + 1)
        pilot = rng.choice(N, min(100, N // 10), replace=False)
        lam = np.cov(y[pilot], f[pilot], ddof=1)[0, 1] / max(f[pilot].var(ddof=1), 1e-12)
        S2 = float((y[pilot] - lam * f[pilot]).var(ddof=1))
        for c, b in zip(curve, [b for b in BUDGETS if 2 * b <= N]):
            fig_rows.append(dict(model=m, n=b, w_classical=c["mean_width_classical"],
                                 w_certified=c["mean_width_ppi"],
                                 coverage_classical=c["coverage_classical"],
                                 coverage_certified=c["coverage_ppi"],
                                 n_rep=c["n_rep"], pool_N=N,
                                 n_pred_pm2pct=required_budget(S2, N, 0.02, ALPHA)))
    df2 = pd.DataFrame(fig_rows)
    df2.to_csv("results/budget_curves.csv", index=False)
    summ = []
    for m in FOUND:
        d = df2[df2.model == m].sort_values("n")
        hit = d[d.w_certified <= 0.04]
        y_m, f_m, _ = campaign_arrays(m, 0.0, "full")
        r400 = d[d.n == 400]
        summ.append(dict(model=m, pool_N=int(d.pool_N.iloc[0]),
                         n_empirical_pm2pct=int(hit.n.iloc[0]) if len(hit) else -1,
                         n_pred_pm2pct=int(d.n_pred_pm2pct.iloc[0]),
                         w_certified_at_500=float(np.interp(500, d.n, d.w_certified)),
                         rho_pool=float(np.corrcoef(y_m, f_m)[0, 1]),
                         ratio_n400=float(r400.w_certified.iloc[0] / r400.w_classical.iloc[0])))
    pd.DataFrame(summ).to_csv("results/budgets_summary.csv", index=False)
    print("budgets_summary includes rho_pool; re-run the Figure 2 cell to overlay real campaigns on panel d")
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    for m in FOUND:
        d = df2[df2.model == m]
        ax.plot(d.n, d.w_certified, "o-", ms=3, lw=1, label=m)
    ax.set_xscale("log"); ax.set_xlabel("DFT verification budget n")
    ax.set_ylabel("mean certified 95% CI width")
    ax.legend(fontsize=5.5, ncol=2); fig.tight_layout()
    fig.savefig("results/fig3_budget_real.png", dpi=400)
    print(df2.groupby("model").n_pred_pm2pct.first().rename(
        "Corollary-1 predicted budget for +/-2% (from 100-label pilot)"))


budgets_summary includes rho_pool; re-run the Figure 2 cell to overlay real campaigns on panel d


/tmp/ipykernel_935/1950121719.py:47: UserWarning: The figure layout has changed to tight
  ax.legend(fontsize=5.5, ncol=2); fig.tight_layout()


model
ALIGNN       2191
CGCNN        2318
CHGNet       1723
M3GNet       2176
MACE         1923
MEGNet       2257
MatterSim    1243
ORB           576
ORB MPtrj    1438
SevenNet     1698
Name: Corollary-1 predicted budget for +/-2% (from 100-label pilot), dtype: int64


### Experiment E3 — paired certified-yield rankings (fills Table 2)
Adjacent pairs in the CONFIG order plus any pair whose certified order flips the error-based (formation-energy MAE) order; Holm-corrected.

In [10]:
def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(running, 1.0)
    return adj

_need = [n for n in ("QUICK_MODE", "ppi_mean_ci", "DATA_OK") if n not in globals()]
if _need:
    print("Run earlier cells first (top to bottom). Missing:", ", ".join(_need))
    print("  -> run the imports cell, the CONFIG cell, then the DATA LOADING cell.")
elif not DATA_OK:
    print("E3 skipped: complete the data-loading cell first.")
else:
    df = DF["full"]
    # model e_form - DFT e_form == each_pred - each_true (identical algebraically),
    # so the MAE is computable under both loading paths:
    mae = {m: float((EACH_PRED["full"][m] - df[truth_col]).abs().mean())
           for m in FOUND}
    err_order = sorted(FOUND, key=mae.get)      # best (lowest MAE) first
    pd.DataFrame([dict(model=m, mae_eV_per_atom=mae[m]) for m in err_order]
                 ).to_csv("results/model_mae.csv", index=False)
    # Table 2 protocol: pairs adjacent in the ERROR-BASED (MAE) order
    pairs = [(err_order[i], err_order[i + 1]) for i in range(len(err_order) - 1)]
    tau, rows = 0.0, []
    for A, B in pairs:
        epA, epB = EACH_PRED["full"][A], EACH_PRED["full"][B]
        ok = epA.notna() & epB.notna() & df[truth_col].notna()
        fA = (epA[ok] <= tau).to_numpy(float); fB = (epB[ok] <= tau).to_numpy(float)
        y = (df.loc[ok, truth_col] <= tau).to_numpy(float)
        union = (fA + fB) > 0
        yu, fAu, fBu = y[union], fA[union], fB[union]
        M_ = len(yu); rng = np.random.default_rng(BASE_SEED + 2)
        ests, zs = [], []
        for _ in range(max(N_REP // 5, 20)):
            idx = rng.choice(M_, min(N_LABEL, M_ // 2), replace=False)
            ci = ppi_yield_diff_ci(yu[idx], fAu[idx], fBu[idx], fAu, fBu, alpha=ALPHA)
            ests.append((ci.estimate, ci.lower, ci.upper))
        est = np.median([e[0] for e in ests]); lo = np.median([e[1] for e in ests])
        hi = np.median([e[2] for e in ests]); se = (hi - lo) / (2 * 1.96)
        from math import erf, sqrt
        p = 2 * (1 - 0.5 * (1 + erf(abs(est / max(se, 1e-12)) / sqrt(2))))
        rows.append(dict(pair=f"{A} vs {B}", mae_A=mae[A], mae_B=mae[B],
                         dq=est, lo=lo, hi=hi, p=p,
                         err_order=("A<B" if mae[A] < mae[B] else "B<A")))
    t2 = pd.DataFrame(rows); t2["p_holm"] = holm(t2.p)
    t2["agrees"] = np.where(
        (t2.lo > 0) | (t2.hi < 0),
        np.where((t2.dq > 0) == (t2.err_order == "A<B"), "yes", "NO (flip)"), "n.s.")
    t2.to_csv("results/table2_raw.csv", index=False)
    print(t2.to_string(index=False))

    # Budget sensitivity (strengthening): same pairs, same estimator, larger
    # shared budget; separate file so the record file stays byte-identical.
    rows2 = []
    for A, B in pairs:
        epA, epB = EACH_PRED["full"][A], EACH_PRED["full"][B]
        ok2 = epA.notna() & epB.notna() & df[truth_col].notna()
        fA = (epA[ok2] <= tau).to_numpy(float); fB = (epB[ok2] <= tau).to_numpy(float)
        y2 = (df.loc[ok2, truth_col] <= tau).to_numpy(float)
        union = (fA + fB) > 0
        yu, fAu, fBu = y2[union], fA[union], fB[union]
        M_ = len(yu); rng = np.random.default_rng(BASE_SEED + 2)
        ests = []
        for _ in range(max(N_REP // 5, 20)):
            idx = rng.choice(M_, min(RANK_BUDGET2, M_ // 2), replace=False)
            ci = ppi_yield_diff_ci(yu[idx], fAu[idx], fBu[idx], fAu, fBu, alpha=ALPHA)
            ests.append((ci.estimate, ci.lower, ci.upper))
        est = np.median([e[0] for e in ests]); lo = np.median([e[1] for e in ests])
        hi = np.median([e[2] for e in ests]); se = (hi - lo) / (2 * 1.96)
        from math import erf, sqrt
        p = 2 * (1 - 0.5 * (1 + erf(abs(est / max(se, 1e-12)) / sqrt(2))))
        rows2.append(dict(n_label=RANK_BUDGET2, pair=f"{A} vs {B}",
                          dq=est, lo=lo, hi=hi, p=p,
                          err_order=("A<B" if mae[A] < mae[B] else "B<A")))
    t2b = pd.DataFrame(rows2); t2b["p_holm"] = holm(t2b.p)
    t2b["agrees"] = np.where(
        (t2b.lo > 0) | (t2b.hi < 0),
        np.where((t2b.dq > 0) == (t2b.err_order == "A<B"), "yes", "NO (flip)"), "n.s.")
    t2b.to_csv("results/table2_budget_sensitivity.csv", index=False)
    n_sig_500 = int((t2.p_holm < 0.05).sum()); n_sig_2 = int((t2b.p_holm < 0.05).sum())
    flips = int((t2b.agrees == "NO (flip)").sum())
    print(f"\nBudget sensitivity at n={RANK_BUDGET2}: {n_sig_2}/9 pairs Holm-significant "
          f"(vs {n_sig_500}/9 at n=500); certified reversals of the error order: {flips}.")
    if QUICK_MODE:
        print("\n=== PILOT MODE: do not paste; re-run with QUICK_MODE=False. ===")
    print("\n--- LaTeX rows for Table 2"
          + (" (PILOT -- not for the paper)" if QUICK_MODE else " (paste into main.tex)") + " ---")
    for _, r in t2.iterrows():
        print(f"{r.pair} & {r.dq:+.4f} & [{r.lo:+.4f}, {r.hi:+.4f}] & "
              f"{r.p_holm:.3f} & {r.err_order} & {r.agrees} \\\\")


                 pair    mae_A    mae_B        dq        lo       hi        p err_order   p_holm agrees
     MatterSim vs ORB 0.026045 0.030844  0.036532  0.012002 0.061083 0.003526       A<B 0.028208    yes
     ORB vs ORB MPtrj 0.030844 0.044705  0.002243 -0.022474 0.027148 0.859362       A<B 1.000000   n.s.
ORB MPtrj vs SevenNet 0.044705 0.046376 -0.005882 -0.030319 0.018712 0.638180       A<B 1.000000   n.s.
     SevenNet vs MACE 0.046376 0.054662  0.013773 -0.009533 0.037010 0.246042       A<B 0.984167   n.s.
       MACE vs CHGNet 0.054662 0.061093  0.023147 -0.001235 0.047238 0.061221       A<B 0.327058   n.s.
     CHGNet vs M3GNet 0.061093 0.072012 -0.022873 -0.045416 0.001216 0.054510       A<B 0.327058   n.s.
     M3GNet vs ALIGNN 0.072012 0.091582  0.062585  0.033994 0.091318 0.000019       A<B 0.000168    yes
     ALIGNN vs MEGNet 0.091582 0.128212  0.039354  0.010148 0.069003 0.008763       A<B 0.061341    yes
      MEGNet vs CGCNN 0.128212 0.134969 -0.007938 -0.037019 0.02

### Experiments E4 (stratified allocation, fills Supplementary Table 2) and E5 (robustness)

In [11]:
_need = [n for n in ("QUICK_MODE", "ppi_mean_ci", "DATA_OK", "campaign_arrays")
         if n not in globals()]
if _need:
    print("Run earlier cells first (top to bottom). Missing:", ", ".join(_need))
    print("  -> run imports, CONFIG, DATA LOADING, then the E1 cell (it defines")
    print("     campaign_arrays, which this cell reuses).")
elif not DATA_OK:
    print("E4/E5 skipped: complete the data-loading cell first.")
else:
    # E4: strata = model-confidence bins on |E_hull_pred - tau|; Neyman vs
    # uniform at the same total budget, for three models spanning the MAE range.
    _mae = {m: float((EACH_PRED["full"][m] - DF["full"][truth_col]).abs().mean())
            for m in FOUND}
    _order = sorted(FOUND, key=_mae.get)
    e4_rows, tau = [], 0.0
    for m in sorted({_order[0], _order[len(_order) // 2], _order[-1]}):
        y, f, N = campaign_arrays(m, tau, "full")
        ep = EACH_PRED["full"][m]; ok = ep.notna() & DF["full"][truth_col].notna()
        conf = np.abs(ep[ok].to_numpy(float)[ep[ok].to_numpy(float) <= tau] - tau)
        bins = np.quantile(conf, [0, 0.33, 0.66, 1.0]); bins[-1] += 1e-9
        sid = np.digitize(conf, bins[1:-1])
        rng = np.random.default_rng(BASE_SEED + 3)
        pilot = rng.choice(len(y), 100, replace=False)
        sds = np.array([max((y[pilot][sid[pilot] == k] -
                             f[pilot][sid[pilot] == k]).std(ddof=1)
                            if (sid[pilot] == k).sum() > 2 else y.std(), 1e-6)
                        for k in range(3)])
        Ns = np.array([(sid == k).sum() for k in range(3)])
        alloc = neyman_allocation(Ns, sds, N_LABEL)
        w_strat, w_unif = [], []
        for _ in range(max(N_REP // 5, 20)):
            strata = []
            for k in range(3):
                ids = np.where(sid == k)[0]
                if len(ids) < 2:            # stratum too thin in this pool -> skip it
                    continue
                take = min(max(alloc[k], 2), len(ids))
                pick = rng.choice(ids, take, replace=False)
                strata.append({"y": y[pick], "f_l": f[pick], "f_pool": f[ids]})
            if len(strata) < 2:             # need >=2 usable strata to stratify
                w_strat, w_unif = None, None
                break
            cs = stratified_ppi_ci(strata, alpha=ALPHA)
            idx = rng.choice(len(y), N_LABEL, replace=False)
            cu = ppi_mean_ci(y[idx], f[idx], f, alpha=ALPHA)
            w_strat.append(cs.upper - cs.lower); w_unif.append(cu.upper - cu.lower)
        ratio = float(np.mean(w_strat) / np.mean(w_unif))
        e4_rows.append(dict(scheme="confidence tertiles", model=m, n=N_LABEL,
                            pilot=100, n_rep=max(N_REP // 5, 20), width_ratio=ratio))
        print(f"E4 [{m}]: stratified/uniform width ratio = {ratio:.3f}")
    # Second stratification scheme (strengthening): composition arity
    import re as _re
    for m in sorted({_order[0], _order[len(_order) // 2], _order[-1]}):
        y, f, N = campaign_arrays(m, tau, "full")
        ep = EACH_PRED["full"][m]; ok = ep.notna() & DF["full"][truth_col].notna()
        flagged_mask = ep[ok].to_numpy(float) <= tau
        forms = DF["full"].loc[ok, "formula"].to_numpy()[flagged_mask]
        nel = np.array([len(_re.findall(r"[A-Z][a-z]?", str(x))) for x in forms])
        sid = np.clip(nel, 2, 4) - 2          # strata: binary, ternary, quaternary+
        rng = np.random.default_rng(BASE_SEED + 3)
        pilot = rng.choice(len(y), 100, replace=False)
        sds = np.array([max((y[pilot][sid[pilot] == k] -
                             f[pilot][sid[pilot] == k]).std(ddof=1)
                            if (sid[pilot] == k).sum() > 2 else y.std(), 1e-6)
                        for k in range(3)])
        Ns = np.array([(sid == k).sum() for k in range(3)])
        alloc = neyman_allocation(Ns, sds, N_LABEL)
        w_strat, w_unif = [], []
        for _ in range(max(N_REP // 5, 20)):
            strata = []
            for k in range(3):
                ids = np.where(sid == k)[0]
                if len(ids) < 2:            # stratum too thin in this pool -> skip it
                    continue
                take = min(max(alloc[k], 2), len(ids))
                pick = rng.choice(ids, take, replace=False)
                strata.append({"y": y[pick], "f_l": f[pick], "f_pool": f[ids]})
            if len(strata) < 2:             # need >=2 usable strata to stratify
                w_strat, w_unif = None, None
                break
            cs = stratified_ppi_ci(strata, alpha=ALPHA)
            idx = rng.choice(len(y), N_LABEL, replace=False)
            cu = ppi_mean_ci(y[idx], f[idx], f, alpha=ALPHA)
            w_strat.append(cs.upper - cs.lower); w_unif.append(cu.upper - cu.lower)
        if w_strat and w_unif:
            ratio = float(np.mean(w_strat) / np.mean(w_unif))
            e4_rows.append(dict(scheme="composition arity", model=m, n=N_LABEL,
                                pilot=100, n_rep=max(N_REP // 5, 20), width_ratio=ratio))
            print(f"E4 [{m}] arity strata: stratified/uniform width ratio = {ratio:.3f}")
        else:
            print(f"E4 [{m}] arity strata: skipped (a composition-arity stratum "
                  f"was too thin in this pool)")
    pd.DataFrame(e4_rows).to_csv("results/tableS2_raw.csv", index=False)

    # E5 (strengthened): robustness variants, now REPLICATED with coverage
    # against census truth instead of a single seeded draw.
    e5_rows = []
    m = _order[0]
    reps5 = max(N_REP // 5, 20)
    for dtau in (-0.01, 0.0, 0.01):
        y2, f2, N2 = campaign_arrays(m, tau + dtau, "full")
        true2 = int(y2.sum()); rng = np.random.default_rng(BASE_SEED + 4)
        widths, covered = [], 0
        for _ in range(reps5):
            idx = rng.choice(N2, N_LABEL, replace=False)
            ci = ppi_mean_ci(y2[idx], f2[idx], f2, alpha=ALPHA)
            widths.append(ci.upper - ci.lower)
            covered += (ci.lower <= true2 / N2 <= ci.upper)
        e5_rows.append(dict(variant=f"margins tau{dtau:+.2f}", model=m, pool_N=N2,
                            n_rep=reps5, mean_width=float(np.mean(widths)),
                            coverage=covered / reps5))
    yb, fb, Nb = campaign_arrays(m, tau, "full")
    fb_bin = np.ones_like(fb); trueb = int(yb.sum())
    rng = np.random.default_rng(BASE_SEED + 5)
    widths, covered = [], 0
    for _ in range(reps5):
        idx = rng.choice(Nb, N_LABEL, replace=False)
        ci = ppi_mean_ci(yb[idx], fb_bin[idx], fb_bin, alpha=ALPHA)
        widths.append(ci.upper - ci.lower)
        covered += (ci.lower <= trueb / Nb <= ci.upper)
    e5_rows.append(dict(variant="binary calls tau+0.00", model=m, pool_N=Nb,
                        n_rep=reps5, mean_width=float(np.mean(widths)),
                        coverage=covered / reps5))
    pd.DataFrame(e5_rows).to_csv("results/e5_robustness.csv", index=False)
    print("E5 saved (replicated):", len(e5_rows), "variants -> results/e5_robustness.csv")


E4 [CGCNN]: stratified/uniform width ratio = 0.994


E4 [CHGNet]: stratified/uniform width ratio = 0.994


E4 [MatterSim]: stratified/uniform width ratio = 0.942
E5 saved: 4 variants -> results/e5_robustness.csv


### EXPORT — one file to upload for paper completion
Run this last. It writes `results/run_metadata.json` (proving which protocol
ran) and packs every result into **`results/paper_results.zip`** — upload that
single file to the chat and the remaining red placeholders can be filled and
verified from it.

In [12]:
import os, json, zipfile, sys, platform, datetime, hashlib
os.makedirs("results", exist_ok=True)
meta = dict(
    QUICK_MODE=globals().get("QUICK_MODE"), N_REP=globals().get("N_REP"),
    N_LABEL=globals().get("N_LABEL"), BUDGETS=globals().get("BUDGETS"),
    TAUS=globals().get("TAUS"), POOLS=globals().get("POOLS"),
    BASE_SEED=globals().get("BASE_SEED"), MARGIN_SCALE=globals().get("MARGIN_SCALE"),
    SEED_ROBUST=globals().get("SEED_ROBUST"), RANK_BUDGET2=globals().get("RANK_BUDGET2"),
    mbd_version=globals().get("MBD_VERSION"), notebook_rev=globals().get("NOTEBOOK_BUILD","v2-strengthened-2026-07-03"),
    models=list(globals().get("FOUND", [])), load_path=globals().get("LOAD_PATH"),
    python=sys.version.split()[0], platform=platform.platform(),
    timestamp=datetime.datetime.now().isoformat(timespec="seconds"))
try:
    import numpy, pandas
    meta["numpy"] = numpy.__version__; meta["pandas"] = pandas.__version__
except Exception:
    pass

# data files the paper depends on (run_metadata.json is written last, separately)
expected = ["table1_raw.csv", "table2_raw.csv", "tableS2_raw.csv",
            "fig2_synthetic_validation.png", "fig2_synthetic_summary.csv",
            "fig2_budget_curve_synthetic.csv",
            "table2_budget_sensitivity.csv", "e6_seed_robustness.csv",
            "budget_curves.csv", "budgets_summary.csv", "model_mae.csv",
            "e5_robustness.csv", "fig3_budget_real.png"]
present = [f for f in expected if os.path.isfile(os.path.join("results", f))]
missing = [f for f in expected if f not in present]

# sha256 manifest over every present data file, then write metadata
meta["file_sha256"] = {f: hashlib.sha256(
    open(os.path.join("results", f), "rb").read()).hexdigest() for f in present}
with open("results/run_metadata.json", "w") as fh:
    json.dump(meta, fh, indent=2)

to_pack = present + ["run_metadata.json"]
with zipfile.ZipFile("results/paper_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in to_pack:
        z.write(os.path.join("results", f), f)
print(f"packed {len(to_pack)} files into results/paper_results.zip")
if missing:
    print("MISSING (run the corresponding experiment cells, then re-run me):", missing)
else:
    print("complete" + (" -- but this is a QUICK_MODE PILOT bundle; set"
          " QUICK_MODE=False and re-run cells 7-12 for the paper protocol"
          if globals().get("QUICK_MODE") else
          " -- PAPER-PROTOCOL bundle; upload results/paper_results.zip"))


packed 9 files into results/paper_results.zip
complete -- PAPER-PROTOCOL bundle; upload results/paper_results.zip


### Reconciliation checklist (after `QUICK_MODE=False` completes)
1. `results/table1_raw.csv` -> Table 1 rows (LaTeX printed above); Supplementary Table 1 = the coverage column across all configs.
2. `results/budget_curves.csv` + `results/fig3_budget_real.png` -> Fig. 3; report Corollary-1 predicted vs empirical budgets.
3. `results/table2_raw.csv` -> Table 2 rows.
4. `results/tableS2_raw.csv` -> Supplementary Table 2.
5. Replace every red placeholder in `main.tex`/`supplementary.tex`, then reconcile the abstract, title and contribution claims with what the numbers actually show. If naive overstatement is negligible or coverage fails anywhere, the framing changes — see RUNBOOK flag-back triggers.